# TakeMeter — NBA Discourse Classifier
**Fine-tune DistilBERT to classify r/nba posts as `analysis`, `hot_take`, or `reaction`.**

### Quick Start
1. Set runtime to **T4 GPU**: Runtime → Change runtime type → T4 GPU → Save
2. Run Section 1 (upload CSV + define labels)
3. Run Section 2 (split & tokenize)
4. Run Section 5 (Groq baseline — add your API key first)
5. Run Section 3 (fine-tune)
6. Run Section 4 (evaluate fine-tuned model)
7. Run Section 6 (side-by-side comparison + export)


## Section 1 — Setup, Label Map, and Data Upload

In [ ]:
# Install / import dependencies
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers", "datasets", "scikit-learn", "groq"], check=False)

import os, json, csv, random
import numpy as np
import pandas as pd
from collections import Counter


In [ ]:
# ── LABEL MAP — edit if you change the taxonomy ──────────────────────────────
LABEL2ID = {"analysis": 0, "hot_take": 1, "reaction": 2}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)
print("Labels:", LABEL2ID)


In [ ]:
# ── UPLOAD YOUR CSV ───────────────────────────────────────────────────────────
# The notebook will prompt you to pick a file from your computer.
# Upload nba_takes_labeled.csv (the file committed to your repo).
from google.colab import files
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]
print(f"Uploaded: {csv_filename}")

df = pd.read_csv(csv_filename)
print(df["label"].value_counts())
print(f"Total rows: {len(df)}")


## Section 2 — Split and Tokenize

In [ ]:
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

# 70 / 15 / 15 split — stratified to preserve label distribution
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42)

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")
print("\nTrain distribution:")
print(train_df["label"].value_counts())
print("\nTest distribution:")
print(test_df["label"].value_counts())


In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts, max_length=128):
    return tokenizer(
        list(texts),
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

train_enc = tokenize(train_df["text"])
val_enc   = tokenize(val_df["text"])
test_enc  = tokenize(test_df["text"])

train_labels = [LABEL2ID[l] for l in train_df["label"]]
val_labels   = [LABEL2ID[l] for l in val_df["label"]]
test_labels  = [LABEL2ID[l] for l in test_df["label"]]

print("Tokenization complete.")


## Section 3 — Fine-Tune DistilBERT

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class NBADataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = NBADataset(train_enc, train_labels)
val_dataset   = NBADataset(val_enc,   val_labels)
test_dataset  = NBADataset(test_enc,  test_labels)

BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID
).to(device)

# ── HYPERPARAMETERS ───────────────────────────────────────────────────────────
# Learning rate 2e-5: standard for DistilBERT fine-tuning on small datasets.
# 3 epochs: enough passes to learn on 140 training examples without overfitting.
# Linear warmup over 10% of steps helps stabilize early training.
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 3
WARMUP_RATIO  = 0.1

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps   = len(train_loader) * NUM_EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f"Training for {NUM_EPOCHS} epochs | {total_steps} total steps | {warmup_steps} warmup steps")


In [ ]:
def evaluate_loader(loader, model, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = outputs.logits.argmax(dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return all_preds, all_labels

# Training loop
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    val_preds, val_true = evaluate_loader(val_loader, model, device)
    val_acc = sum(p == t for p, t in zip(val_preds, val_true)) / len(val_true)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  loss={avg_loss:.4f}  val_acc={val_acc:.4f}")

print("\nTraining complete!")


## Section 4 — Evaluate Fine-Tuned Model

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")  # for Colab

test_preds, test_true = evaluate_loader(test_loader, model, device)

ft_accuracy = accuracy_score(test_true, test_preds)
ft_f1_macro = f1_score(test_true, test_preds, average="macro")

label_names = [ID2LABEL[i] for i in range(NUM_LABELS)]

print(f"Fine-Tuned Model — Test Accuracy: {ft_accuracy:.4f}  Macro F1: {ft_f1_macro:.4f}")
print()
print(classification_report(test_true, test_preds, target_names=label_names))


In [ ]:
# ── CONFUSION MATRIX ──────────────────────────────────────────────────────────
cm = confusion_matrix(test_true, test_preds)
print("Confusion Matrix (rows=true, cols=predicted):")
print(f"{'':>12}", "  ".join(f"{l:>12}" for l in label_names))
for i, row in enumerate(cm):
    print(f"{label_names[i]:>12}", "  ".join(f"{v:>12}" for v in row))

# Save PNG
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_LABELS)); ax.set_xticklabels(label_names, rotation=30, ha="right")
ax.set_yticks(range(NUM_LABELS)); ax.set_yticklabels(label_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix — Fine-Tuned DistilBERT")
for i in range(NUM_LABELS):
    for j in range(NUM_LABELS):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
print("Saved confusion_matrix.png")


In [ ]:
# ── WRONG PREDICTIONS ─────────────────────────────────────────────────────────
# Shows all misclassified test examples — pick 3 to analyze in your README
test_texts = list(test_df["text"])
wrong = [(test_texts[i], ID2LABEL[test_true[i]], ID2LABEL[test_preds[i]])
         for i in range(len(test_true)) if test_true[i] != test_preds[i]]

print(f"\nTotal wrong predictions: {len(wrong)} / {len(test_true)}")
print("="*70)
for text, true, pred in wrong:
    print(f"TRUE: {true:10}  PRED: {pred:10}")
    print(f"TEXT: {text[:120]}...")
    print("-"*70)


In [ ]:
# ── CONFIDENCE SCORES (sample classifications) ────────────────────────────────
import torch.nn.functional as F

model.eval()
sample_texts = test_texts[:5]
sample_enc = tokenizer(sample_texts, padding="max_length", truncation=True,
                       max_length=128, return_tensors="pt").to(device)

with torch.no_grad():
    logits = model(**sample_enc).logits
    probs  = F.softmax(logits, dim=-1)

print("Sample Classifications:")
print(f"{'Text (truncated)':50} {'Predicted':12} {'Confidence':10}")
print("-"*75)
for i, text in enumerate(sample_texts):
    pred_id   = probs[i].argmax().item()
    confidence = probs[i][pred_id].item()
    true_label = ID2LABEL[test_true[i]]
    print(f"{text[:48]:50} {ID2LABEL[pred_id]:12} {confidence:.2%}  (true: {true_label})")


## Section 5 — Groq Zero-Shot Baseline

**Before running:** Add your Groq API key.
- Click the 🔑 icon in the left sidebar
- Add a secret named `GROQ_API_KEY`
- Enable notebook access


In [ ]:
from google.colab import userdata
import groq, time

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
client = groq.Groq(api_key=GROQ_API_KEY)

SYSTEM_PROMPT = """You are a classifier for NBA Reddit discourse quality.
Classify each post into exactly one of these three labels:

- analysis: The post makes a structured argument supported by statistics, historical comparisons, tactical observations, or verifiable evidence. Evidence is specific and would stand on its own even if the opinion framing were removed.
- hot_take: A bold, confident opinion stated without supporting evidence, or with evidence so thin or cherry-picked that it functions as decoration rather than reasoning. The post asserts rather than argues.
- reaction: An immediate emotional response to a specific recent event. Expresses a feeling in the moment with little to no structured argument.

Rules:
1. Reply with ONLY the label name — no punctuation, no explanation.
2. The label must be exactly one of: analysis, hot_take, reaction
3. If a post cites a stat but is primarily conclusion-first with the stat as decoration, label it hot_take.
4. If a post is emotional about a specific moment but mentions a number, label it reaction.
"""

def classify_groq(text):
    try:
        resp = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": text}
            ],
            max_tokens=10,
            temperature=0
        )
        return resp.choices[0].message.content.strip().lower()
    except Exception as e:
        print(f"Error: {e}")
        return "error"

# Run baseline on test set
baseline_preds_raw = []
print(f"Running Groq baseline on {len(test_texts)} test examples...")
for i, text in enumerate(test_texts):
    pred = classify_groq(text)
    baseline_preds_raw.append(pred)
    if (i+1) % 5 == 0:
        print(f"  {i+1}/{len(test_texts)}")
    time.sleep(0.3)  # rate limiting

print("Done!")


In [ ]:
# Parse and score baseline
valid_labels = set(LABEL2ID.keys())
baseline_preds, baseline_true = [], []
unparseable = 0

for i, pred in enumerate(baseline_preds_raw):
    if pred in valid_labels:
        baseline_preds.append(LABEL2ID[pred])
        baseline_true.append(test_true[i])
    else:
        unparseable += 1
        print(f"Unparseable: '{pred}'")

baseline_accuracy = accuracy_score(baseline_true, baseline_preds)
baseline_f1_macro = f1_score(baseline_true, baseline_preds, average="macro")

print(f"\nUnparseable responses: {unparseable}")
print(f"Baseline Accuracy: {baseline_accuracy:.4f}  Macro F1: {baseline_f1_macro:.4f}")
print()
print(classification_report(baseline_true, baseline_preds, target_names=label_names))


## Section 6 — Side-by-Side Comparison + Export

In [ ]:
# ── SIDE-BY-SIDE COMPARISON ───────────────────────────────────────────────────
from sklearn.metrics import f1_score, precision_score, recall_score
import json

label_names = [ID2LABEL[i] for i in range(NUM_LABELS)]

def per_class_metrics(true, preds, labels):
    result = {}
    for i, label in enumerate(labels):
        true_bin = [1 if t == i else 0 for t in true]
        pred_bin = [1 if p == i else 0 for p in preds]
        result[label] = {
            "precision": precision_score(true_bin, pred_bin, zero_division=0),
            "recall":    recall_score(true_bin, pred_bin, zero_division=0),
            "f1":        f1_score(true_bin, pred_bin, zero_division=0),
        }
    return result

ft_per_class  = per_class_metrics(test_true, test_preds, range(NUM_LABELS))
bl_per_class  = per_class_metrics(baseline_true, baseline_preds, range(NUM_LABELS))

print("="*65)
print(f"{'Model':<20} {'Accuracy':>10} {'Macro F1':>10}")
print("="*65)
print(f"{'Groq Zero-Shot':<20} {baseline_accuracy:>10.4f} {baseline_f1_macro:>10.4f}")
print(f"{'DistilBERT FT':<20} {ft_accuracy:>10.4f} {ft_f1_macro:>10.4f}")
print()
print(f"{'Label':<14} {'BL P':>6} {'BL R':>6} {'BL F1':>6}  {'FT P':>6} {'FT R':>6} {'FT F1':>6}")
print("-"*60)
for i, lname in enumerate(label_names):
    bl = bl_per_class[i]
    ft = ft_per_class[i]
    print(f"{lname:<14} {bl['precision']:>6.3f} {bl['recall']:>6.3f} {bl['f1']:>6.3f}  "
          f"{ft['precision']:>6.3f} {ft['recall']:>6.3f} {ft['f1']:>6.3f}")


In [ ]:
# ── EXPORT evaluation_results.json ───────────────────────────────────────────
results = {
    "baseline": {
        "model": "groq/llama-3.3-70b-versatile (zero-shot)",
        "accuracy": baseline_accuracy,
        "macro_f1": baseline_f1_macro,
        "per_class": {label_names[i]: bl_per_class[i] for i in range(NUM_LABELS)}
    },
    "fine_tuned": {
        "model": "distilbert-base-uncased (fine-tuned, 3 epochs, lr=2e-5)",
        "accuracy": ft_accuracy,
        "macro_f1": ft_f1_macro,
        "per_class": {label_names[i]: ft_per_class[i] for i in range(NUM_LABELS)},
        "confusion_matrix": cm.tolist()
    }
}

with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved evaluation_results.json")
print("\nNow download both files from the Files panel (right-click → Download):")
print("  • evaluation_results.json")
print("  • confusion_matrix.png")
print("Commit both to your GitHub repo.")


## Section 7 — Deployed Interface (Stretch Feature)
Run this cell to launch a Gradio interface for interactive classification.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=False)
import gradio as gr
import torch.nn.functional as F

def classify_post(text):
    model.eval()
    enc = tokenizer(text, padding="max_length", truncation=True,
                    max_length=128, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**enc).logits
        probs  = F.softmax(logits, dim=-1)[0]
    pred_id    = probs.argmax().item()
    confidence = probs[pred_id].item()
    label      = ID2LABEL[pred_id]
    breakdown  = {ID2LABEL[i]: float(probs[i]) for i in range(NUM_LABELS)}
    result = f"**Predicted Label:** `{label}`\n**Confidence:** {confidence:.1%}\n\n"
    result += "| Label | Probability |\n|-------|------------|\n"
    for l, p in sorted(breakdown.items(), key=lambda x: -x[1]):
        result += f"| {l} | {p:.1%} |\n"
    return result

demo = gr.Interface(
    fn=classify_post,
    inputs=gr.Textbox(lines=4, placeholder="Paste an NBA post or comment here..."),
    outputs=gr.Markdown(),
    title="TakeMeter — NBA Discourse Classifier",
    description="Classify r/nba posts as **analysis**, **hot_take**, or **reaction**.",
    examples=[
        ["Jokic's assist-to-turnover ratio this postseason is 4.8:1 — best for any center in playoff history since the merger."],
        ["LeBron is literally the GOAT and anyone who disagrees is just hating. There is no debate."],
        ["WE JUST WON THE CHAMPIONSHIP. I'm crying and I don't care who knows. 20 years of pain and we finally did it!!"],
    ]
)
demo.launch(share=True)
